# Batch Normalization

신경망이 깊어질수록 각 층으로 들어가는 값의 분포가 불안정해질 수 있고, 그 결과 학습이 느리거나 흔들릴 수 있다.

Batch Normalization은 미니배치 단위의 평균과 분산을 사용해 값을 정규화한 뒤,
필요하면 다시 학습 가능한 스케일과 이동을 적용하는 층이다.

- 각 층으로 들어가는 값의 스케일을 너무 제멋대로 두지 않고
- 학습을 좀 더 안정적으로 진행하도록 돕는 장치
- 경우에 따라 더 큰 학습률 사용이나 빠른 수렴에 도움을 줄 수 있는 장치

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

## 1. BatchNorm

PyTorch의 `BatchNorm1d`는 학습 중 현재 배치의 평균과 분산으로 정규화하고,
동시에 running mean / running variance를 저장해 두었다가 평가 시 사용한다.  
또한 기본적으로 affine 파라미터를 가져서, 정규화 뒤에도 학습 가능한 스케일과 이동을 적용할 수 있다.

1. 학습 시: 현재 배치 통계량 사용
2. 평가 시: 학습 중 누적한 running 통계량 사용
3. 필요하면 학습 가능한 `gamma`, `beta`로 다시 조정

### 입력 정규화와 Batch Normalization의 차이

둘 다 **스케일을 다루는 것** 이라는 공통점이 있지만 역할이 다르다.

입력 정규화(Standardization 등)는
- 데이터셋의 입력값 자체를 미리 정리하는 전처리 단계이다.

Batch Normalization은
- 모델 내부 층 사이에 들어가며
- 학습 중 미니배치 통계량을 사용해 중간 표현을 조정한다.

즉,
- 입력 정규화는 데이터 준비 단계
- BatchNorm은 모델 내부의 층
이라고 구분하면 된다.

## 2. train 모드와 eval 모드 차이

BatchNorm은 dropout과 마찬가지로 `train()` / `eval()`에 따라 동작이 달라진다.

- `train()`에서는 현재 배치의 평균/분산 사용
- `eval()`에서는 저장된 running mean / running variance 사용

따라서 BatchNorm이 있는 모델도 검증/테스트/추론 시 `model.eval()`이 꼭 필요하다.

## 3. 실습

MLP에서는 보통 다음과 같이 자주 쓴다.

`Linear -> BatchNorm1d -> ReLU`

즉,
1. 선형 변환 결과를 만들고
2. 그 값을 BatchNorm으로 정리한 뒤
3. 활성화 함수를 통과시키는 흐름이다.

In [ ]:
# 데이터 생성
X, y = make_moons(n_samples=1000, noise=0.35, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_val = torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32)


## 정리

1. Batch Normalization은 층 사이의 값을 미니배치 기준으로 정규화하는 층이다.
2. 학습 시에는 현재 배치의 평균/분산을 사용하고, 평가 시에는 running 통계량을 사용한다. 
3. 입력 정규화는 데이터 전처리이고, BatchNorm은 모델 내부 층이다.
4. BatchNorm은 학습을 더 안정적으로 만드는 데 도움을 줄 수 있다.
5. BatchNorm이 있는 모델도 검증/추론 시 `model.eval()`이 매우 중요하다. 